In [43]:
print("Training process for Random Forest...\n\n🛢️ Random Forest Pipeline: SMOTE -> Random Forest")
rf_pipeline = ImbPipeline([
    (
        "smote", 
        SMOTE(random_state=42)
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        )
    )
])
print("✅ Done!\n")

print("🔎 Random Forest hyperparameter search...")
rf_params = {
    "model__n_estimators": randint(150, 700),
    "model__max_depth": [10, 15, 20, 25, 30, None],
    "model__min_samples_split": randint(5, 50),
    "model__min_samples_leaf": randint(2, 15),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5]
}
print("✅ Done!\n")

print("🚀 Initializing cross-validation object...")
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
print("✅ Done!\n")

print("🎲 Initializing a search manager object...")
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_params,
    n_iter = 30,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
print("✅ Done!\n")

Training process for Random Forest...

🛢️ Random Forest Pipeline: SMOTE -> Random Forest
✅ Done!

🔎 Random Forest hyperparameter search...
✅ Done!

🚀 Initializing cross-validation object...
✅ Done!

🎲 Initializing a search manager object...
✅ Done!



In [ ]:
print("🎓 Fitting the search manager...")
rf_search.fit(X_train_rf, y_train_rf)

print(f"Best hyperparameters for Random Forest model are:\n{rf_search.best_params_}")
print(f"The best score is:{rf_search.best_score_}\n")

best_rf = rf_search.best_estimator_
print("💾 Saving best performing Random Forest model...")

In [60]:
try:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    joblib.dump(best_rf, f"models/random_forest_best_{timestamp}.joblib")
    joblib.dump(rf_search, f"models/rf_random_search_{timestamp}.joblib")
    print("✅ Random Forest model with best performance score was successfully saved!")
    
except Exception as e:
    print(f"❌ Something went wrong during the model serialization: {e}")

✅ Random Forest model with best performance score was successfully saved!


In [61]:
# ============================================================
# SHARED EVAL HELPER
# ============================================================
# Used by every LOF / IF scoring step below so ROC-AUC / PR-AUC
# aren't hand-computed and hand-printed in a slightly different
# way in every cell. Always pass scores already oriented
# higher = more anomalous (see create_anomaly_predictions).

def evaluate_scores(y_true, y_score, label):
    roc = roc_auc_score(y_true, y_score)
    pr = average_precision_score(y_true, y_score)
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(f"ROC-AUC: {roc:.6f}")
    print(f"PR-AUC:  {pr:.6f}")
    return roc, pr


def create_anomaly_predictions(model, X, df, extra_columns=None):
    """
    Shared prediction helper for any sklearn-style outlier model
    exposing score_samples (IsolationForest, LOF with novelty=True).
    anomaly_score is oriented higher = more anomalous.
    """
    data = {
        "txId": df["txId"].values,
        "time_step": df["time_step"].values,
        "anomaly_score": -model.score_samples(X),
    }
    if extra_columns:
        data.update(extra_columns)
    return pd.DataFrame(data)


In [62]:
def create_lof_pipeline(
    n_neighbors=50,
    contamination=0.05,
    metric="euclidean",
    p=2,
    leaf_size=30,
):
    return Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "lof",
            LocalOutlierFactor(
                n_neighbors=n_neighbors,
                contamination=contamination,
                metric=metric,
                p=p,
                leaf_size=leaf_size,
                novelty=True,
                n_jobs=-1,
            ),
        ),
    ])

In [63]:
## 1. LOF Baseline

baseline_lof = create_lof_pipeline(
    n_neighbors=50,
    contamination=0.05,
    metric="euclidean",
    p=2,
    leaf_size=30,
)

baseline_lof.fit(X_train)

baseline_predictions = create_anomaly_predictions(
    baseline_lof, X_validation, validation_df
)

baseline_eval = baseline_predictions.merge(
    validation_labels,
    on="txId",
    how="inner",
)

baseline_roc_auc, baseline_pr_auc = evaluate_scores(
    baseline_eval["is_illicit"].astype(int),
    baseline_eval["anomaly_score"],
    "Baseline LOF",
)


Baseline LOF
ROC-AUC: 0.473827
PR-AUC:  0.076293


In [64]:
## 2. LOF Hyperparameter Search

n_neighbors_values = [
    75,
    100,
    125,
    150,
    175,
    200,
    250,
    300,
    350,
]

lof_search_results = []

for n_neighbors in n_neighbors_values:

    print(f"Testing n_neighbors={n_neighbors}")

    lof_model = create_lof_pipeline(
        n_neighbors=n_neighbors,
        contamination=0.05,
        metric="euclidean",
        p=2,
        leaf_size=30,
    )

    lof_model.fit(X_train)

    search_predictions = create_anomaly_predictions(
        lof_model, X_validation, validation_df
    )

    search_eval = search_predictions.merge(
        validation_labels,
        on="txId",
        how="inner",
    )

    roc_auc, pr_auc = evaluate_scores(
        search_eval["is_illicit"].astype(int),
        search_eval["anomaly_score"],
        f"LOF search — n_neighbors={n_neighbors}",
    )

    lof_search_results.append({
        "n_neighbors": n_neighbors,
        "metric": "euclidean",
        "leaf_size": 30,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    })

lof_search_results_df = (
    pd.DataFrame(lof_search_results)
    .sort_values(
        by=["pr_auc", "roc_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(lof_search_results_df)


Testing n_neighbors=75
LOF search — n_neighbors=75
ROC-AUC: 0.506001
PR-AUC:  0.083006
Testing n_neighbors=100
LOF search — n_neighbors=100
ROC-AUC: 0.532648
PR-AUC:  0.098443
Testing n_neighbors=125
LOF search — n_neighbors=125
ROC-AUC: 0.508838
PR-AUC:  0.105038
Testing n_neighbors=150
LOF search — n_neighbors=150
ROC-AUC: 0.492528
PR-AUC:  0.098484
Testing n_neighbors=175
LOF search — n_neighbors=175
ROC-AUC: 0.481320
PR-AUC:  0.095613
Testing n_neighbors=200
LOF search — n_neighbors=200
ROC-AUC: 0.472817
PR-AUC:  0.093928
Testing n_neighbors=250
LOF search — n_neighbors=250
ROC-AUC: 0.463485
PR-AUC:  0.091758
Testing n_neighbors=300
LOF search — n_neighbors=300
ROC-AUC: 0.457451
PR-AUC:  0.090182
Testing n_neighbors=350
LOF search — n_neighbors=350
ROC-AUC: 0.457143
PR-AUC:  0.089772


,n_neighbors,metric,leaf_size,roc_auc,pr_auc
0,125,euclidean,30,0.508838,0.105038
1,150,euclidean,30,0.492528,0.098484
2,100,euclidean,30,0.532648,0.098443
3,175,euclidean,30,0.481320,0.095613
4,200,euclidean,30,0.472817,0.093928
5,250,euclidean,30,0.463485,0.091758
6,300,euclidean,30,0.457451,0.090182
7,350,euclidean,30,0.457143,0.089772
8,75,euclidean,30,0.506001,0.083006


In [65]:
## 3. Selected LOF Configuration

best_lof_params = lof_search_results_df.iloc[0]

print("Selected configuration:")
print(
    f"n_neighbors: {int(best_lof_params['n_neighbors'])}"
)
print(
    f"metric:      {best_lof_params['metric']}"
)
print(
    f"leaf_size:   {int(best_lof_params['leaf_size'])}"
)
print(
    f"ROC-AUC:     {best_lof_params['roc_auc']:.6f}"
)
print(
    f"PR-AUC:      {best_lof_params['pr_auc']:.6f}"
)

Selected configuration:
n_neighbors: 125
metric:      euclidean
leaf_size:   30
ROC-AUC:     0.508838
PR-AUC:      0.105038


In [66]:
## 4. Selected LOF — Validation Evaluation
selected_lof = create_lof_pipeline(
    n_neighbors=int(best_lof_params["n_neighbors"]),
    contamination=0.05,
    metric=best_lof_params["metric"],
    p=2,
    leaf_size=int(best_lof_params["leaf_size"]),
)

selected_lof.fit(X_train)

print("Selected LOF trained.")
print("Training shape:", X_train.shape)

validation_predictions = create_anomaly_predictions(
    selected_lof, X_validation, validation_df
)

validation_eval = validation_predictions.merge(
    validation_labels,
    on="txId",
    how="inner",
)

print("Validation transactions:", len(validation_predictions))
print("Labeled validation transactions:", len(validation_eval))
display(validation_eval.head())

selected_roc_auc, selected_pr_auc = evaluate_scores(
    validation_eval["is_illicit"].astype(int),
    validation_eval["anomaly_score"],
    "Selected LOF validation performance",
)


Selected LOF trained.
Training shape: (136265, 165)
Validation transactions: 30680
Labeled validation transactions: 7829


,txId,time_step,anomaly_score,is_illicit
0,54751137,35,2.485928,0
1,67576672,35,1.040865,0
2,69767012,35,1.578661,0
3,70384401,35,1.095432,1
4,67603017,35,1.250184,1


Selected LOF validation performance
ROC-AUC: 0.508838
PR-AUC:  0.105038


In [67]:
## 5. Final LOF Model

# ============================================================
# FINAL LOF MODEL
# ============================================================

# Use the selected hyperparameters from validation search.
final_lof = create_lof_pipeline(
    n_neighbors=int(best_lof_params["n_neighbors"]),
    contamination=0.05,
    metric=best_lof_params["metric"],
    p=2,
    leaf_size=int(best_lof_params["leaf_size"]),
)

# Fit the complete pipeline.
final_lof.fit(X_train_full)

print("Final LOF trained successfully.")

Final LOF trained successfully.


In [68]:
## 7. Save Final LOF Model

model_path = Path("models/lof/lof_final.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(final_lof, model_path)
print(f"Final LOF saved to: {model_path}")

Final LOF saved to: models/lof/lof_final.joblib


In [69]:
# ============================================================
# FINAL LOF — TEMPORAL TEST EVALUATION
# ============================================================

test_lof_predictions = create_anomaly_predictions(
    final_lof, X_test, test_df
)

test_lof_eval = test_lof_predictions.merge(
    test_labels,
    on="txId",
    how="inner",
)

temporal_lof_roc, temporal_lof_pr = evaluate_scores(
    test_lof_eval["is_illicit"].astype(int),
    test_lof_eval["anomaly_score"],
    "FINAL LOF — TEMPORAL TEST",
)


FINAL LOF — TEMPORAL TEST
ROC-AUC: 0.460650
PR-AUC:  0.047858


In [70]:
print("📚 Train Isolation Forest on training period (time_step 1-34)\n")
if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)


if_model.fit(X_train)


print("🌱 Isolation Forest trained")

📚 Train Isolation Forest on training period (time_step 1-34)

🌱 Isolation Forest trained


In [71]:
val_predictions_df = create_anomaly_predictions(
    if_model, X_validation, validation_df
)

val_eval_df = val_predictions_df.merge(
    validation_labels,
    on="txId",
    how="inner"
)

print("🕵🏻 Validation labeled transactions:", len(val_eval_df))
display(val_eval_df.head())


🕵🏻 Validation labeled transactions: 7829


,txId,time_step,anomaly_score,is_illicit
0,54751137,35,0.453791,0
1,67576672,35,0.421571,0
2,69767012,35,0.439560,0
3,70384401,35,0.355382,1
4,67603017,35,0.347754,1


In [72]:
if_val_roc, if_val_pr = evaluate_scores(
    val_eval_df["is_illicit"].astype(int),
    val_eval_df["anomaly_score"],
    "Validation IF ranking",
)


Validation IF ranking
ROC-AUC: 0.092923
PR-AUC:  0.045936


In [73]:
final_if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination=0.05,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

final_if_model.fit(X_train_full)

joblib.dump(final_if_model, "models/isolation_forest/isolation_forest_best.joblib")
print("Final IF trained on time steps 1-41.")

# ============================================================
# FINAL IF — TEMPORAL TEST EVALUATION
# ============================================================

test_predictions_df = create_anomaly_predictions(
    final_if_model, X_test, test_df
)

test_eval_df = test_predictions_df.merge(
    test_labels,
    on="txId",
    how="inner"
)

print("🧾 Test labeled transactions:", len(test_eval_df))

# anomaly_score is already oriented higher = more anomalous
# (see create_anomaly_predictions) -- do not negate again here.
temporal_if_roc, temporal_if_pr = evaluate_scores(
    test_eval_df["is_illicit"].astype(int),
    test_eval_df["anomaly_score"],
    "FINAL IF — TEMPORAL TEST",
)


Final IF trained on time steps 1-41.
🧾 Test labeled transactions: 8841
FINAL IF — TEMPORAL TEST
ROC-AUC: 0.280879
PR-AUC:  0.029305
